# AML/KYC Compliance Flagging Agent — Notebook Walkthrough

**HBF2212 — Artificial Intelligence in Finance, Project 2**

This notebook demonstrates the AML/KYC compliance agent's core pipeline
step by step, outside the Streamlit interface. It clones the project's
public GitHub repository and runs the same `rules_engine.py` module used
by the deployed app, so the logic shown here is identical to production.

- **Live app:** https://aml-kyc-agent-ryut5cxe9.streamlit.app  *(replace with your final URL)*
- **GitHub repo:** https://github.com/mjt040305-cloud/aml-kyc-agent

**Pipeline stages covered in this notebook:**
1. Read — load transaction data
2. Analyse — run the AML rules engine (4 risk categories, 6 rules)
3. Decide/flag — risk scoring and bucketing
4. Human oversight checkpoint — simulated compliance officer review
5. Output — final compliance report

The full deployed app additionally orchestrates this pipeline as a
**LangGraph state graph** (`agent_graph.py`) with a genuine `interrupt()`
pause at the human oversight step — see the final section of this
notebook, and the live app itself, for that orchestration layer in
action.

## 1. Setup — clone the repository and install dependencies

In [ ]:
!git clone -q https://github.com/mjt040305-cloud/aml-kyc-agent.git
%cd aml-kyc-agent
!pip install -q -r requirements.txt

## 2. Read — load transaction data

We use the bundled synthetic dataset (`sample_transactions.csv`), which contains normal activity plus deliberately embedded suspicious patterns for testing.

In [ ]:
import pandas as pd
from rules_engine import analyse_transactions, DEFAULT_CONFIG, SEVERITY_ICON, CATEGORIES

df = pd.read_csv("sample_transactions.csv")
print(f"Loaded {len(df)} transactions")
df.head(10)

## 3. Analyse + Decide/Flag — run the AML rules engine

Each transaction is scored across four risk categories (Customer, Transaction, Geographic, Behavioural) and bucketed into None / Low / Medium / High risk. `DEFAULT_CONFIG` holds the AML thresholds — the same dict a compliance officer can override from the deployed app's sidebar.

In [ ]:
print("Default AML rule configuration:")
for k, v in DEFAULT_CONFIG.items():
    print(f"  {k}: {v}")

analysed = analyse_transactions(df)
analysed[["transaction_id", "customer_id", "amount", "risk_score", "risk_bucket"]].head(15)

In [ ]:
summary = analysed["risk_bucket"].value_counts()
print("Risk distribution:")
print(summary)

### Inspect the risk breakdown for the highest-scoring transaction

This shows the explainability that makes the rule-based approach auditable: every category contribution and every triggered rule is visible, not a black-box score.

In [ ]:
top = analysed.iloc[0]
print(f"Transaction {top['transaction_id']} — Customer {top['customer_id']} — ${top['amount']:,.2f}")
print(f"Overall risk score: {top['risk_score']} ({top['risk_bucket']})")
print()
print("Category breakdown:")
for cat, score in top["category_scores"].items():
    print(f"  {cat}: {score}")
print()
print("AML rules triggered:")
for rule in top["triggered_rules"]:
    icon = SEVERITY_ICON[rule["severity"]]
    print(f"  {icon} {rule['label']} ({rule['category']}) — {rule['reason']}")

## 4. Human Oversight Checkpoint (simulated)

In the deployed Streamlit app, this step is a genuine LangGraph `interrupt()` — the agent pauses and cannot proceed until a compliance officer records a decision. In this notebook we simulate that same decision-making step programmatically so the pipeline can run end-to-end without an interactive UI.

In [ ]:
pending = analysed[analysed["risk_bucket"].isin(["High", "Medium"])].copy()
print(f"{len(pending)} transaction(s) require human review before this pipeline can produce final output.")
pending[["transaction_id", "customer_id", "amount", "risk_bucket", "flag_reasons"]]

In [ ]:
# Simulated compliance officer decisions.
# In the live app, a human enters these through the UI and the LangGraph
# agent literally cannot proceed past this point without them.
import random
random.seed(1)

decisions = {}
for _, row in pending.iterrows():
    decision = "Escalate to SAR filing" if row["risk_bucket"] == "High" else "Approve (false positive)"
    decisions[row["transaction_id"]] = {
        "status": decision,
        "reviewer": "J. Musina (compliance officer)",
        "notes": "Reviewed against customer KYC file and transaction history.",
    }

for txn_id, d in decisions.items():
    print(f"{txn_id}: {d['status']}")

## 5. Output — final compliance report

In [ ]:
analysed["review_status"] = analysed["transaction_id"].map(
    lambda tid: decisions.get(tid, {}).get("status", "Not required")
)
analysed["reviewed_by"] = analysed["transaction_id"].map(
    lambda tid: decisions.get(tid, {}).get("reviewer", "")
)
analysed["reviewer_notes"] = analysed["transaction_id"].map(
    lambda tid: decisions.get(tid, {}).get("notes", "")
)

final_report = analysed[[
    "transaction_id", "customer_id", "amount", "date", "risk_bucket",
    "risk_score", "flag_reasons", "review_status", "reviewed_by", "reviewer_notes"
]]

final_report.to_csv("notebook_compliance_report.csv", index=False)
print("Saved notebook_compliance_report.csv")
final_report.head(15)

## 6. Agent orchestration — running the real LangGraph agent

Sections 2-5 called `rules_engine.analyse_transactions()` directly for
clarity. The deployed Streamlit app instead runs this same function
*inside* a LangGraph `StateGraph` (`agent_graph.py`):

```
START -> analyse -> human_review (interrupt) -> output -> END
```

Unlike the earlier sandbox used to draft this project, **Google Colab has
internet access**, so this section installs `langgraph` for real and
**actually executes the graph** — including triggering and resuming the
genuine `interrupt()` pause at the human oversight node. This is not a
simulation: if the pause did not work, the assertions below would fail.

In [ ]:
!pip install -q langgraph langchain-core

In [ ]:
from agent_graph import build_agent_graph, run_pipeline, resume_pipeline
import uuid

graph = build_agent_graph()
thread_id = str(uuid.uuid4())

result = run_pipeline(graph, df.to_dict("records"), thread_id, rules_config=DEFAULT_CONFIG)

print("Pipeline status:", result["status"])
assert result["status"] == "awaiting_review", "Expected the graph to pause for human review"
print(f"Graph genuinely paused at human_review — {len(result['pending_transactions'])} transaction(s) awaiting a decision.")

In [ ]:
# Confirm the pause is real: at this point no final report exists yet -
# the graph's state is parked at the human_review node.
print("Pending transactions (graph is paused here):")
for t in result["pending_transactions"]:
    print(f"  {t['transaction_id']} - {t['customer_id']} - risk {t['risk_score']} ({t['risk_bucket']})")

In [ ]:
# Resume the graph with simulated compliance officer decisions.
# This is the ONLY way execution can reach the output node - there is no
# other code path in agent_graph.py that produces a final_report.
graph_decisions = {
    t["transaction_id"]: {
        "status": "Escalate to SAR filing" if t["risk_bucket"] == "High" else "Approve (false positive)",
        "reviewer": "J. Musina (compliance officer)",
        "notes": "Reviewed via LangGraph resume in Colab.",
    }
    for t in result["pending_transactions"]
}

final = resume_pipeline(graph, graph_decisions, thread_id)
print("Pipeline status after resume:", final["status"])
assert final["status"] == "complete"
print(f"Graph resumed and reached the output node — {len(final['final_report'])} transactions in final report.")

In [ ]:
import pandas as pd
pd.DataFrame(final["final_report"])[[
    "transaction_id", "customer_id", "risk_bucket", "risk_score", "review_status", "reviewed_by"
]].head(15)

### What this demonstrates

- `run_pipeline()` returned `status: "awaiting_review"` — the graph
  genuinely stopped at `human_review` and returned control to this
  notebook, rather than completing in one call.
- Nothing in `agent_graph.py` can produce a `final_report` without going
  through `resume_pipeline()` — there is no bypass.
- After supplying decisions and calling `resume_pipeline()`, the graph
  continued to `output` and produced the final, reviewed report.

This is the same orchestration the deployed Streamlit app uses — the
only difference is that the app collects the officer's decisions through
an interactive UI instead of a Python dict, and holds the pause across
Streamlit reruns rather than across notebook cells.